In [7]:
export MODEL="unsloth/gemma-4-26B-A4B-it-GGUF:UD-IQ4_NL"
# to set sudo sysctl iogpu.wired_limit_mb=14800. Needs to be set on termimal outside for pure GPU
sysctl iogpu.wired_limit_mb # to check

iogpu.wired_limit_mb: 15000


In [5]:
llama-fit-params -hf $MODEL

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.009 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [8]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="1024 2048"  
UBATCH_SIZES="64 128 256 512"
  
echo "Testing different batch/ubatch combinations..."  
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do  
        # Get fitted parameters  
        #OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub | grep -o '\-c [0-9]* \-ngl [0-9]*')  

        OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub 2>&1)
        
        #echo "Raw output: $OUTPUT"  # Debug line  
        
        # Extract context and GPU layers more robustly  
        CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
        NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
        
        if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
            echo "b: $b, ub: $ub, ctx: $CTX, ngl: $NGL"  
        else  
            echo "No valid parameters found"  
        fi  
    done  
done  

Testing different batch/ubatch combinations...
b: 1024, ub: 64, ctx: 44544, ngl: -1
b: 2048, ub: 64, ctx: 44544, ngl: -1
b: 1024, ub: 128, ctx: 40960, ngl: -1
b: 2048, ub: 128, ctx: 40960, ngl: -1
b: 1024, ub: 256, ctx: 34048, ngl: -1
b: 2048, ub: 256, ctx: 34048, ngl: -1
b: 1024, ub: 512, ctx: 18432, ngl: -1
b: 2048, ub: 512, ctx: 18432, ngl: -1


In [5]:
# Pure GPU
# memory around 15GiB, cpu barely used
# try different ub sizes, use default batch size
# watch out -fa does not default 1 for llama-bench (but is enabled for llama-server, rather if set to auto, looks like it will be enabled)
llama-bench -hf $MODEL -ngl 99 -fa 1 -ub 256,512 -t 8 -p 1000,2000 -n 128,256

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.024 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [51]:
# Increase fitt to 512 (leave 512 MiB of memory left on target device)
llama-fit-params -hf $MODEL -ub 128 -fitt 512
# result: -c 56832 -ngl -1, but this context size memory causes chibaboom in llama-server
# which is strange, also this changes to 30k the next time it was ran

load_backend: loaded BLAS backend from /opt/homebrew/Cellar/ggml/0.9.11/libexec/libggml-blas.so
ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.055 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSet

In [ ]:
llama-server -hf $MODEL \
    --threads-http 1 --threads 8 --no-mmproj --reasoning off \
    --no-warmup -ub 128 -ngl -1 -np 1 --host 0.0.0.0 # -np 1 parameter is important